In [93]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
# import importlib
# import src.utils.cleaner

# importlib.reload(src.utils.cleaner)

<module 'src.utils.cleaner' from 'c:\\Users\\hkand\\streaming\\sales_streaming_analytics\\src\\utils\\cleaner.py'>

In [94]:
import sys
import os

home = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(home)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType
from src.utils.json_parser import parse_json
from src.utils.time_utils import date_day_weekOfMonth
from src.utils.cleaner import good_exp_records, bad_exp_records
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog")
checkpoints = dbutils.widgets.get("checkpoints_dir")

df = spark.readStream.table(f"{catalog}.brz.expenses_raw")

df = df.select("value")

schema = StructType([
    StructField("expense_id", IntegerType(), True),
    StructField("employee_id", LongType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("expense_type", StringType(), True),
    StructField("expense_amount", LongType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

parsed_df = parse_json(spark, df, "value", schema)

parsed_df = (parsed_df.select("parsed.expense_id", "parsed.employee_id",
                                "parsed.region_id", "parsed.expense_type",
                                "parsed.expense_amount", "parsed.event_time"))

good_exp_df = good_exp_records(spark, parsed_df)

bad_exp_df = bad_exp_records(spark, parsed_df)

good_exp_df = date_day_weekOfMonth(spark, good_exp_df, "event_time")

bad_exp_df = date_day_weekOfMonth(spark, bad_exp_df, "event_time")

good_query = (good_exp_df.writeStream
                .format("delta")
                .option("checkpointLocation", f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints}/slv_checkpoints/expenses_checkpoint")
                .trigger(availableNow = True)
                .outputMode("append")
                .table(f"{catalog}.slv.expenses"))

bad_query = (bad_exp_df.writeStream
                .format("delta")
                .option("checkpointLocation", f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints}/brz_checkpoints/bad_expenses_checkpoint")
                .trigger(availableNow = True)
                .outputMode("append")
                .table(f"{catalog}.brz.bad_expenses_records"))

In [99]:
spark.sql("select * from sales_streaming_dev.slv.sales order by sales_id limit 5;")

,sales_id,employee_id,region_id,product_id,quantity,sales_amount,event_time,event_date,day,week_of_month,processed_time
0,1,3,5,6,1,625,2026-05-29 11:18:09.900940,2026-05-29,Friday,5,2026-05-30 12:07:39.906
1,1,9,1,28,2,2900,2026-05-30 15:54:54.631036,2026-05-30,Saturday,5,2026-05-30 16:23:38.555
2,2,10,5,6,1,625,2026-05-29 11:18:09.900988,2026-05-29,Friday,5,2026-05-30 12:07:39.906
3,2,1,2,26,1,375,2026-05-30 15:54:54.631119,2026-05-30,Saturday,5,2026-05-30 16:23:38.555
4,3,6,3,41,2,16900,2026-05-29 11:18:09.901021,2026-05-29,Friday,5,2026-05-30 12:07:39.906
